# Telugu Movie Worldwide Gross Prediction — Machine Learning Model

This notebook trains a **real supervised ML regression model** using the uploaded Telugu movie dataset. It predicts worldwide gross directly (₹ crore), rather than averaging hero/director recovery percentages.

**Important:** The current workbook has a small number of usable historical films. The model is a prototype, and test metrics can vary a lot with such a small dataset. Only rows marked `Released` with valid positive budgets and nonnegative worldwide gross are used for training. Unclear and upcoming records are flagged and excluded from training.

## 1. Install and import libraries

In [ ]:
!pip -q install pandas openpyxl scikit-learn joblib

import io, re, warnings
import numpy as np
import pandas as pd
from google.colab import files
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
print('Libraries ready.')

## 2. Upload your movie dataset
Upload `telugu_movie_prediction_dataset.xlsx` (or your corrected workbook). Do not type missing values from memory.

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise ValueError('Upload the Excel dataset first.')
filename = next(iter(uploaded))
xls = pd.ExcelFile(io.BytesIO(uploaded[filename]))
sheet = 'Movie_Dataset' if 'Movie_Dataset' in xls.sheet_names else xls.sheet_names[0]
df = pd.read_excel(io.BytesIO(uploaded[filename]), sheet_name=sheet)
df.columns = [str(c).strip() for c in df.columns]
display(df.head())
print('Rows:', len(df), '| Columns:', list(df.columns))

## 3. Validate and prepare the data
The model learns to predict **Worldwide_Gross_₹_crore** from budget, hero, and director. Upcoming/unclear movies are not training examples because their target gross is unknown or unverified.

In [ ]:
REQUIRED = ['Movie', 'Hero', 'Director', 'Budget_₹_crore', 'Worldwide_Gross_₹_crore', 'Release_Status']
missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}. Please use the expected workbook columns.')

for c in ['Budget_₹_crore', 'Worldwide_Gross_₹_crore']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['Release_Status_clean'] = df['Release_Status'].astype(str).str.strip().str.lower()
valid = (
    df['Release_Status_clean'].eq('released') &
    df['Budget_₹_crore'].notna() & (df['Budget_₹_crore'] > 0) &
    df['Worldwide_Gross_₹_crore'].notna() & (df['Worldwide_Gross_₹_crore'] >= 0) &
    df['Hero'].notna() & df['Director'].notna()
)
train_df = df.loc[valid].copy()
excluded_df = df.loc[~valid].copy()
print(f'Usable training rows: {len(train_df)} / {len(df)}')
print(f'Excluded/flagged rows: {len(excluded_df)}')
if len(train_df) < 8:
    raise ValueError('Too few usable records for even a basic train/test evaluation. Add more verified released films.')
display(excluded_df[['Movie','Release_Status','Budget_₹_crore','Worldwide_Gross_₹_crore','Notes'] if 'Notes' in df.columns else ['Movie','Release_Status','Budget_₹_crore','Worldwide_Gross_₹_crore']])

## 4. Train a real ML regression model
We use a Random Forest regressor. Hero and director are categorical inputs; budget is numeric. Multi-hero strings are split into individual hero names and multi-hot encoded. A held-out test set provides a basic check of generalization.

In [ ]:
def split_names(value):
    if pd.isna(value): return []
    return [x.strip() for x in re.split(r';|,|\\band\\b', str(value), flags=re.I) if x.strip()]

# Normalize common NTR naming variants while retaining the original source table.
def normalize_hero_name(x):
    s = str(x).strip()
    aliases = {'ntr':'Jr NTR', 'jr ntr':'Jr NTR', 'junior ntr':'Jr NTR'}
    return aliases.get(s.lower(), s)

train_df['Hero_list'] = train_df['Hero'].apply(lambda x: [normalize_hero_name(h) for h in split_names(x)])
train_df['Director_clean'] = train_df['Director'].astype(str).str.strip()
mlb = MultiLabelBinarizer()
hero_matrix = mlb.fit_transform(train_df['Hero_list'])
hero_features = pd.DataFrame(hero_matrix, columns=[f'hero__{h}' for h in mlb.classes_], index=train_df.index)
X = pd.concat([train_df[['Budget_₹_crore']].rename(columns={'Budget_₹_crore':'budget_crore'}), hero_features,
               pd.get_dummies(train_df['Director_clean'], prefix='director', dtype=int)], axis=1)
y = train_df['Worldwide_Gross_₹_crore'].astype(float)

# With very small datasets, keep a test set but interpret scores cautiously.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = RandomForestRegressor(n_estimators=400, min_samples_leaf=2, max_features=0.9, random_state=42)
model.fit(X_train, y_train)
test_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, test_pred)
rmse = mean_squared_error(y_test, test_pred) ** 0.5
r2 = r2_score(y_test, test_pred) if len(y_test) >= 2 else float('nan')
metrics = pd.DataFrame([{'Metric':'MAE (₹ crore; lower is better)','Value':mae},
                        {'Metric':'RMSE (₹ crore; lower is better)','Value':rmse},
                        {'Metric':'R² (higher can be better; unstable on tiny test sets)','Value':r2},
                        {'Metric':'Training rows','Value':len(X_train)}, {'Metric':'Test rows','Value':len(X_test)}])
display(metrics)
display(pd.DataFrame({'Actual gross (₹ crore)':y_test, 'Predicted gross (₹ crore)':np.round(test_pred,1),
                      'Absolute error (₹ crore)':np.round(np.abs(y_test.to_numpy()-test_pred),1)}, index=y_test.index))
print('This is trained ML: model.fit(X_train, y_train) learned parameters from historical examples.')

## 5. Ask the agent for a new movie
Enter the hero, director, and upcoming budget. The trained model predicts worldwide gross directly. If a hero/director was never seen during training, the notebook flags that limitation; unknown names cannot magically provide a learned effect.

In [ ]:
hero_input = input('Enter hero name (for multiple heroes, separate with semicolons): ').strip()
director_input = input('Enter director name: ').strip()
budget_input = input('Enter upcoming movie budget in ₹ crore: ').strip()
try:
    budget_input = float(budget_input)
except ValueError:
    raise ValueError('Budget must be a number in ₹ crore.')
if not hero_input or not director_input or budget_input <= 0:
    raise ValueError('Enter a hero, director, and a positive budget.')

def build_input_row(hero, director, budget):
    heroes = [normalize_hero_name(h) for h in split_names(hero)]
    row = {c:0 for c in X.columns}
    row['budget_crore'] = budget
    for h in heroes:
        col = f'hero__{h}'
        if col in row: row[col] = 1
    dcol = f'director_{director.strip()}'
    if dcol in row: row[dcol] = 1
    return pd.DataFrame([row], columns=X.columns), heroes

new_X, entered_heroes = build_input_row(hero_input, director_input, budget_input)
predicted_gross = max(0.0, float(model.predict(new_X)[0]))
known_heroes = set(mlb.classes_)
unknown_heroes = [h for h in entered_heroes if h not in known_heroes]
known_directors = set(train_df['Director_clean'])
warnings_list = []
if unknown_heroes: warnings_list.append('Hero(s) not seen in training: ' + ', '.join(unknown_heroes))
if director_input.strip() not in known_directors: warnings_list.append('Director not seen in training: ' + director_input.strip())
display(Markdown(f'## Prediction\\n- **Hero:** {hero_input}\\n- **Director:** {director_input}\\n- **Budget:** ₹{budget_input:,.2f} crore\\n- **ML predicted worldwide gross:** **₹{predicted_gross:,.2f} crore**'))
if warnings_list:
    display(Markdown('**Data limitations**\\n' + '\\n'.join('- ' + w for w in warnings_list)))
display(Markdown('This is a model estimate, not a guarantee. Small datasets and unseen people can make predictions unreliable.'))

## 6. Save the trained model and audit files
Save the fitted model together with the exact feature columns and hero encoder so you can reuse it later. Export excluded records for manual review.

In [ ]:
bundle = {'model': model, 'feature_columns': list(X.columns), 'hero_encoder': mlb,
          'training_rows': train_df[['Movie','Hero','Director','Budget_₹_crore','Worldwide_Gross_₹_crore']].copy(),
          'target': 'Worldwide_Gross_₹_crore', 'features': ['budget_crore','hero indicators','director indicators']}
joblib.dump(bundle, 'telugu_movie_gross_ml_model.joblib')
excluded_df.to_excel('excluded_or_unclear_movie_records.xlsx', index=False)
print('Saved model bundle and excluded-record audit workbook.')
files.download('telugu_movie_gross_ml_model.joblib')
files.download('excluded_or_unclear_movie_records.xlsx')

## Project explanation
- **Type:** Supervised machine learning, regression.
- **Target:** Worldwide gross in ₹ crore.
- **Features:** Budget, hero, director.
- **Algorithm:** Random Forest Regressor.
- **Training:** The model learns from rows using `model.fit(X_train, y_train)`.
- **Evaluation:** Held-out MAE, RMSE, and R².
- **Data integrity:** Upcoming/unclear rows are excluded; no missing gross values are invented.

**Caution:** With only a few dozen films, the model is a working prototype rather than a reliable commercial predictor. Add more verified historical films and test on a larger, chronologically held-out set before making strong claims.